# Knowledge base (ChromaDB)

In [170]:
import numpy as np
import torch
import pandas as pd
import chromadb
from tqdm.notebook import tqdm
from PIL import Image
from transformers import CLIPModel, CLIPProcessor


CSV_PATH = "./knowledge/clean_youtube_watch_log.csv"
THUMBNAIL_PATH = "./knowledge/thumbnails"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# model = model.to("mps" if torch.backends.mps.is_available() else "cpu")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="youtube_videos")

In [171]:
def embed_text(text: str) -> np.ndarray:
    """Embed a text using CLIP"""
    inputs = proc(text=[text], return_tensors="pt", truncation=True, padding=True)  # WARNING! Truncation may lead to loss of information for long texts
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)  # Normalize
    return emb.to("cpu").numpy()


def embed_image(image: Image.Image) -> np.ndarray:
    """Embed an image using CLIP"""
    inputs = proc(images=image, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)  # Normalize
    return emb.to("cpu").numpy()

## Find new videos to eventually add to the collection

In [172]:
df = pd.read_csv(CSV_PATH)
ids_to_add = set(df["video_id"]) - set(collection.get(ids=None)["ids"])
print(f"Found {len(ids_to_add)} new videos to add to the collection.")

for video_id in tqdm(ids_to_add, desc="Building KB"):
    video_data = df[df["video_id"] == video_id].iloc[0]
    user_id = video_data["user_id"]
    title = video_data["video_title"]
    thumbnail = Image.open(f"{THUMBNAIL_PATH}/{video_id}.jpg")

    # Embed text and image, then fuse them (weighted average)
    text_emb = embed_text(title).squeeze()
    img_emb = embed_image(thumbnail).squeeze()
    fused_emb = 0.7 * text_emb + 0.3 * img_emb

    # Add embedding to the collection
    collection.add(
        ids=[video_id],                     # ID
        embeddings=[fused_emb],             # Embedding (fused text + image)
        metadatas=[{"user_id": int(user_id)}]    # Metadata (user_id for filtering during retrieval)
    )

Found 0 new videos to add to the collection.


Building KB: 0it [00:00, ?it/s]

# Building the Agent (LangChain)

In [173]:
from dataclasses import dataclass
from typing import Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from PIL import Image
from typing import Literal
from typing import Optional
from langgraph.types import Command
from langchain.agents import AgentState
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain.messages import ToolMessage
from dataclasses import asdict
from langchain_groq import ChatGroq
from dataclasses import field
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
import re
import uuid
from IPython.display import display, HTML

with open('llm-api-key.txt') as f:
    api_key = f.readline().strip()

# llm = ChatOllama(model="qwen3.5:2B", temperature=0)   # or qwen3:0.6b-q4_K_M
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    # model="llama-3.3-70b-versatile",
    api_key=api_key,
    temperature=0.4,
)
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     temperature=0,
#     api_key=api_key,
#     convert_system_message_to_human=True,
# )

## 1. Data types

In [174]:
@dataclass
class WatchItem:
    video_id: str
    video_title: str
    timestamp: str      # ISO format

    @property
    def thumbnail_url(self) -> str:
        return f'./knowledge/thumbnails/{self.video_id}.jpg'
    
    @classmethod
    def from_row(cls, row) -> "WatchItem":
        return cls(
            video_id=row['video_id'],
            video_title=row['video_title'],
            timestamp=row['watch_date']
        )

    def __repr__(self): # json
        d = asdict(self)
        d["thumbnail_url"] = self.thumbnail_url
        return str(d)

@dataclass
class BiasProfile:
    emotional_tone: Literal["low", "medium", "high"]
    sensationalism: Literal["low", "medium", "high"]
    topical_narrowing: Literal["low", "medium", "high"]
    echo_chamber_effect: Literal["low", "medium", "high"]
    dominant_topics: list[str]
    evidence_titles: list[str]
    overall_profile: str = field(
        metadata={"description": "Some comments from the model explaining why the scores were assigned."}
    )
    confidence: float = field(
        default=0.0,
        metadata={"description": "A confidence score (0-1) indicating how confident the model is in its bias assessment."}
    )

    def __repr__(self):
        return str(asdict(self))

## 2. Type definitions
> **⚠️ Models do NOT have access to these definitions, but tools do!**

In [175]:
# State (short-term memory)
class AgentState(AgentState):
    watch_history: list[WatchItem]                          # @dataclass
    bias_profile: Optional[BiasProfile]
    messages: Annotated[list[BaseMessage], add_messages]    # automatically append

# Context (static configuration)
@dataclass
class AgentContext():
    user_id: int

## 3. Tools

In [176]:
@tool
def retrieve_session(sort_asc: bool, limit: int, runtime: ToolRuntime[AgentContext, AgentState]) -> Command:
    """ Loads a user's watch session in the agent state.
    Args:
        sort_asc (bool): sort oldest first if True
        limit (int): maximum number of videos to retrieve
    Returns:
        Command: Updates the agent state with the retrieved watch history.
    """
    df = pd.read_csv(CSV_PATH)
    df = df[df['user_id'] == runtime.context.user_id]   # Filter by user ID from state
    df = df.sort_values("watch_date", ascending=sort_asc)
    df = df.head(limit) if limit else df
    result = df.apply(WatchItem.from_row, axis=1).tolist()  # Convert to WatchItem list

    return Command(
        update ={
            "watch_history": result,    # Update agent state with the retrieved watch history
            "messages": [ToolMessage(
                content=str(result),
                tool_call_id=runtime.tool_call_id
            )],
        },
    )


@tool
def analyze_bias_profile(runtime: ToolRuntime[AgentContext, AgentState]) -> Command:
    """ Analyzes a user's watch history and loads it in the agent state. 
    Requires a previous run of retrieve_session to have populated the watch_history in the state.
    Returns:
        Command: Updates the agent state with the bias profile.
    """
    watch_history = runtime.state["watch_history"]

    if not watch_history:
        return Command(
            update = {
                "messages": [ToolMessage(
                    content="No watch history found. Please run retrieve_session first.",
                    tool_call_id=runtime.tool_call_id
                )],
            },
        )
    
    global bias_analyzer_agent
    raw = bias_analyzer_agent.invoke(HumanMessage(content="Analyze this watch history for bias: " + str(watch_history)))
    result: BiasProfile = raw["structured_response"]

    return Command(
        update = { 
            "bias_profile": result,
            "messages": [ToolMessage(
                content=asdict(result), # bug! can't use str(...) as would use default __rept__
                tool_call_id=runtime.tool_call_id
            )],
        },
    )


@tool
def find_similar_videos(title_or_thumbnail_url: str, limit: int, runtime: ToolRuntime[AgentContext, AgentState]) -> list[WatchItem]:
    """Invokes the VectorDB to find similar videos by title and/or thumbnail.
    Args:
        title_or_thumbnail_url (str): The title or thumbnail URL of the video to find similarities for.
        limit (int): The number of similar videos to return. Defaults to 5.
    Returns:
        list[WatchItem]: A list of similar videos
    """
    # Embed the input title and thumbnail

    # TODO cross-user search?
    # TODO if title or thumbnail exists, augment query with the corresponding one to improve results
    # TODO check if file exists and manage URL-based thumbnails
    if title_or_thumbnail_url.lower().endswith((".jpg", ".jpeg", ".png")):
        # treat as thumbnail path
        query_vec = embed_image(Image.open(title_or_thumbnail_url))
    else:
        # treat as text
        query_vec = embed_text(title_or_thumbnail_url)

    # Query the vector database for similar videos
    similar_ids = collection.query(
        query_embeddings=query_vec,
        n_results=limit or 5,
        where={"user_id": runtime.context.user_id}
    ).get("ids", [[]])[0]  

    return (
        df[df["video_id"].isin(similar_ids)]
        .apply(WatchItem.from_row, axis=1)
        .tolist()
    )


TOOLS = [retrieve_session, analyze_bias_profile, find_similar_videos]

## 4. Agents

In [177]:
# ---------- Node 1: General-Purpose LLM agent  ----------
SYSTEM_PROMPT = """
You are a YouTube watch-history analysis agent focused on bias, polarization, sensationalism, clickbait, emotional tone, and rabbit-hole effects.

# Rules:
- If you are sure about the user’s intent AND it matches a tool’s purpose, call that tool with appropriately typed arguments. Otherwise, do not call any tool and ask for clarification.
- When asked about bias, polarization, sensationalism, clickbait, emotional tone, or rabbit-hole effects:
  - First ensure the watch history was retrieved using the relative tool.
  - Then call the analyze_bias_profile tool to obtain a structured bias profile.
- Base all claims on factual data from watch history and tool results. For each claim, mention a specific example from the watch history that supports it (e.g., a video title or topic). Avoid vague generalizations.
- Be cautious and do not overclaim ideology; if unsure, say that the evidence is weak or ambiguous and optionally ask for clarification.
- When asked to display an image, respond with the <img src="URL"> tag, where URL is the best image URL from user input, state, memory, or tools.
- Do not call tools multiple times in the same turn; if you need to call another tool, wait for the next turn to do so.
"""
agent = create_agent(
    llm,
    tools=TOOLS,
    state_schema=AgentState,
    context_schema=AgentContext,
    system_prompt=SYSTEM_PROMPT,
    middleware=[ToolCallLimitMiddleware(run_limit=1)],   # Only one tool call per turn
    checkpointer=InMemorySaver(serde=JsonPlusSerializer(
        allowed_msgpack_modules=[
            ("__main__", "WatchItem"),
            ("__main__", "BiasProfile"),
        ]
    )),
)


# ---------- Node 2: Bias Analysis sub-agent  ----------
BIAS_AGENT_PROMPT = """
You analyze a single YouTube watch session and return a structured bias profile and assessment of rabbit-hole effects.

# Rules
- Use only the provided watch-session records; do not assume any additional history.
- Be cautious and conservative in your conclusions.
- Do not overclaim ideology; avoid assigning strong labels without clear evidence.
- If evidence for a bias, ideology, or rabbit-hole is weak or conflicting, explicitly say it is unclear.
- Focus your analysis on:
  - Ideological or political lean (if any).
  - Topic concentration versus diversity.
  - Emotional tone and sensationalism.
  - Escalation or narrowing over time (rabbit-hole behavior).

# Example structuring (optional)
Bias profile:
- Ideological lean: ...
- Dominant topics: ...
- Emotional tone and sensationalism: ...
- Clickbait patterns: ...
- Rabbit-hole effects: ... (for example, increasing extremity or narrowing of topics over the session)
- Confidence / unclear aspects: ...
"""
bias_analyzer_agent = create_agent(
    llm,
    state_schema=AgentState,
    system_prompt=BIAS_AGENT_PROMPT,
    response_format=BiasProfile,
)

# 5. Testing the agents
> **⚠️ WARNING: Don't forget to run cell above!**

In [178]:
user_id = 25
thread_id = uuid.uuid4().hex

print("Agent 🤖 is ready. Type 'exit' to quit.\n")
while True:
    user = HumanMessage(content=input("\nYou: ").strip())
    if user.content == '' or user.content.lower() == "exit":
        break
    
    user.pretty_print()
    for step in agent.stream(
        { "messages" : [user] },
        config={ "configurable": {"thread_id": thread_id} },    # Add memory
        context={"user_id": user_id},                           # Retrieved in tools
    ):
        for update in step.values():
            for m in update.get("messages", []) if update else []:
                m.pretty_print()

Agent 🤖 is ready. Type 'exit' to quit.

================================ Human Message =================================

what is the last three videos i watched
================================== Ai Message ==================================
Tool Calls:
  retrieve_session (fc_321676ea-3370-47f9-864f-a376422895ae)
 Call ID: fc_321676ea-3370-47f9-864f-a376422895ae
  Args:
    limit: 3
    sort_asc: False
================================= Tool Message =================================
Name: retrieve_session

[{'video_id': 'F8wJdvXK5yU', 'video_title': 'SkiVel', 'timestamp': '2018-09-11 17:02:26', 'thumbnail_url': './knowledge/thumbnails/F8wJdvXK5yU.jpg'}, {'video_id': 'YJbS-WA7tVg', 'video_title': 'Lords mobile! What are familiars, And how do you use them', 'timestamp': '2018-09-10 23:37:46', 'thumbnail_url': './knowledge/thumbnails/YJbS-WA7tVg.jpg'}, {'video_id': 'TRa9IMvccjg', 'video_title': 'DILBAR Full Song | Satyameva Jayate | John Abraham Nora Fatehi | Tanishk B Neha Kakkar Ikka Dh

# Debugging

## @tool returning Command

In [179]:
user_id = 25
thread_id = uuid.uuid4().hex

dummy_runtime = ToolRuntime(
    context=AgentContext(user_id=user_id),
    state=AgentState(
        watch_history=[WatchItem.from_row(df[df["user_id"] == user_id].iloc[0])],  # Example state with one video
        bias_profile=None,
        messages=[]
    ),
    tool_call_id="test_call_1",
    config={ "configurable": {"thread_id": thread_id} },
    stream_writer=lambda x: None,
    store=lambda x: None
)

# command = retrieve_session.func(sort_asc=True, limit=5, runtime=dummy_runtime)
command = analyze_bias_profile.func(runtime=dummy_runtime)
command.update["messages"][0].pretty_print()

================================= Tool Message =================================

{'emotional_tone': 'low', 'sensationalism': 'low', 'topical_narrowing': 'low', 'echo_chamber_effect': 'low', 'dominant_topics': [], 'evidence_titles': [], 'overall_profile': 'Insufficient watch-session data was provided; cannot determine ideological lean, dominant topics, emotional tone, sensationalism, or rabbit‑hole effects.', 'confidence': 0.0}
